# Nova training template

Five cells. The notebook holds no training logic: it logs in, clones the repo, loads a YAML config, sets the seed and calls `training/train.py`.

**Before running**
- Kaggle: Settings → Accelerator **GPU T4**, Internet **on**, Add-ons → Secrets → tick `HF_TOKEN`.
- Colab: Runtime → Change runtime type → **T4 GPU**; key icon → `HF_TOKEN` with Notebook access on.

**After a disconnect**: start a new session, set `RESUME = True` in cell 3, and run all cells again.

In [ ]:
# 1. Secrets and login. Tokens come from the platform's secrets panel, never from this cell.
%pip install -q -U trl peft pyyaml

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle has two T4s; use one so runs match Colab

try:                                   # Kaggle
    from kaggle_secrets import UserSecretsClient
    get = UserSecretsClient().get_secret
except ImportError:                    # Colab
    from google.colab import userdata
    get = userdata.get

from huggingface_hub import login, whoami
login(token=get("HF_TOKEN"))
print("Logged in to Hugging Face as", whoami()["name"])

In [ ]:
# 2. Clone the repo. Code comes from Git, not from this notebook.
import subprocess, sys

REPO = "github.com/jagjotsingh244-source/customer-support-SLM.git"
REPO_DIR = os.path.abspath("customer-support-SLM")

try:
    token = get("GITHUB_TOKEN")        # needed only once the repo is private
except Exception:
    token = ""
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.isdir(REPO_DIR):
    cmd = ["git", "-C", REPO_DIR, "pull", "--ff-only"]
else:
    cmd = ["git", "clone", "--depth", "1", url, REPO_DIR]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:            # hide the token if git prints the URL
    raise RuntimeError(result.stderr.replace(token, "***") if token else result.stderr)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Load the settings from the YAML config in the repo.
from training.train import load_config, run

CONFIG = "training/configs/drill.yaml"
RESUME = False    # set to True after a disconnect to continue from the Hub checkpoint

cfg = load_config(CONFIG)
cfg

In [ ]:
# 4. Fix the seed so two runs of the same config are comparable.
from transformers import set_seed

set_seed(cfg["seed"])

In [ ]:
# 5. Train. Watch for ">>> Training starts at step ..." and ">>> Saved checkpoint at step ...".
run(cfg, resume=RESUME)